In [0]:
#1
data = [
    [1, 101, "Laptop", "Electronics", 55000, "2026-01-05", "sakshi@gmail.com"],
    [2, 102, "Phone", "Electronics", 30000, "2026-01-07", "nishi@gmail.com"],
    [3, 103, "Chair", "Furniture", 4500, "2026-01-10", None],
    [4, 104, "Table", "Furniture", None, "2026-01-12", "neha@gmail.com"],
    [5, 105, "Headphones", "Electronics", 2500, "2026-01-15", "amit@gmail.com"],
    [6, 106, "Shoes", "Fashion", 3200, "2026-01-18", None],
    [7, 107, "Watch", "Fashion", 5000, "2026-01-20", "pooja@gmail.com"],
    [8, 108, "Keyboard", "Electronics", 1800, "2026-01-22", "rohit@gmail.com"],
    [8, 108, "Keyboard", "Electronics", 1800, "2026-01-22", "rohit@gmail.com"],
    [9, 109, "Backpack", "Fashion", 2200, "2026-01-25", None],
    [10, 110, "Monitor", "Electronics", 15000, "2026-01-27", "simran@gmail.com"],
    [11, 111, "Sofa", "Furniture", 25000, "2026-02-01", "karan@gmail.com"],
    [12, 112, "Mouse", "Electronics", 900, "2026-02-03", None],
    [13, 113, "Jacket", "Fashion", 4200, "2026-02-05", "meena@gmail.com"],
    [13, 113, "Jacket", "Fashion", 4200, "2026-02-05", "meena@gmail.com"],
    [14, 114, "Desk", "Furniture", 8000, "2026-02-08", "arjun@gmail.com"],
    [15, 115, "Tablet", "Electronics", 22000, "2026-02-10", None]
]

columns = ["order_id", "customer_id", "product", "category", "amount", "order_date", "email"]
df = spark.createDataFrame(data, columns)
df = df.write.mode("overwrite").saveAsTable('cyntexa_dev.sales.ecommerce')

In [0]:
from pyspark.sql import functions as f

df = spark.table("cyntexa_dev.sales.ecommerce")

df = df.dropDuplicates()
df = df.filter(f.col('category') == 'Electronics')

df.show()

In [0]:
#2
df = df.withColumnsRenamed({'product' : 'product_name', 'order_date' : 'date', 'email' : 'customer_email'})

3

- create git folder in databricks
- write code in notebook
- push the notebook in main branch

#Intermediate Tasks

In [0]:
#4
df = spark.table("cyntexa_dev.sales.ecommerce")

df = df.dropna(how='all')
df = df.fillna({'email': 'unknown', 'amount': 0})
df = df.dropDuplicates()
df = df.sort(f.col('order_date').desc())

In [0]:
#5
category_revenue = df.groupBy(f.col('category')).agg(
    f.sum('amount').alias('total_revenue')
)

customer_df = spark.read.table("cyntexa_dev.sales.customers")
df_join = df.join(
    customer_df,
    on=df.customer_id.cast("string") == customer_df.customer_id,
    how="left"
)

df_join.display()
category_revenue.display()

6. 
 order by order_date changed to see the latest `records`


#Advanced Tasks

In [0]:
#7.
from pyspark.sql import functions as f

df = df.withColumn(
    "order_date", f.to_date("order_date", "yyyy-MM-dd")
)

invalid_df = df.filter(
    f.col("amount").isNull() |
    f.col("order_date").isNull()
)

valid_df = df.filter(
    (f.col("amount").isNotNull() & (f.col("amount") > 0))  |
    f.col("order_date").isNotNull()
)
print("Invalid records:")
invalid_df.display()

print("Valid records:")
valid_df.display()

clean_df = (
    df.select("order_id", "customer_id", "product", "category", "amount", "order_date", "email")
)

clean_df = clean_df.withColumn("email",
    f.when(
        (f.col("email").isNull()) |
        (f.lower(f.col("email")) == "unknown"),
        None
    ).otherwise(f.col("email"))
)

clean_df.write.format("delta").mode("overwrite").saveAsTable("cyntexa_dev.sales.orders")

Approach -

create a datafram from a existing table
make the same data format
find invalid records
clean that dataframe
save the cleaned df as a table.

8
branching strategy -

developers create feature branches from dev for development and never push directly to main
after completing a feature the developer pushes the branch and opens a PR into dev
the developer must test the changes and provide a clear PR description
once the PR is approved and checks pass, it can be merged into dev
after integration testing, a PR from dev to main is created main is protected and represents production-ready code, so direct pushes are not allowed

In [0]:
%sql
-- 9
select category, count(customer_id) as total_customer from cyntexa_dev.sales.orders  group by category order by total_customer desc limit 1

In [0]:
%sql
-- month-over-month growth,
SELECT month(order_date) as month,SUM(amount) as total_revenue FROM cyntexa_dev.sales.orders GROUP BY month(order_date);
     
-- average order value trend
SELECT month(order_date) as month,round(AVG(amount),2) as avg_order_value FROM cyntexa_dev.sales.orders GROUP BY month(order_date);
    